# Day 26 In-Class Assignment: SageMaker Setup & Data Loading

**Use this notebook to guide your in-class work on Day 26.**

You will work to:
- Create a "SageMaker AI" classic notebook instance
- Clone your GitHub repo into JupyterLab
- Export model-ready aggregates from Athena to S3
- Load and verify data in SageMaker AI using pandas
- Document your modeling plan

**At the end of class, commit all changes to GitHub and submit this notebook to D2L.**

> **Name (FIRST and LAST):** _delete this text and type your name here_

---

## Learning objectives

By the end of this class period, you should be able to:

- **Launch SageMaker AI classic notebooks** and navigate JupyterLab
- **Clone a GitHub repo** into your notebook environment
- **Export aggregated data from Athena to S3** in CSV format
- **Load and inspect data** in SageMaker AI using pandas
- **Create a modeling plan** tied to your stakeholder question
- **Commit your work** to GitHub

- - -

## Table of contents

1. [SageMaker classic setup](#setup)
2. [GitHub repo clone](#github)
3. [Environment test](#test)
4. [Athena data export](#athena)
5. [Load and verify](#load)
6. [Modeling plan](#plan)
7. [Git commit](#git)

- - -

<a id="setup"></a>

## 1. SageMaker classic setup

**Note**: For simplicity, I'll generally refer to "SageMaker AI classic notebooks" as just "SageMaker notebooks" or "SageMaker" moving forward. The "AI" part seems to be AWS's new branding for SageMaker, but the functionality seems to be the same. Additionally, we will not be using SageMaker Studio, which has additional functionality, but it harder to set up. The "classic notebooks" are simpler and sufficient for our needs.

### Step 1: Create notebook instance

We did this together in class, so **if you followed along, you can skip this section.** Below is for reference.

- AWS Console → Search **"SageMaker AI"** → Select **SageMaker AI**
- Left sidebar → **Notebooks** → **Notebook instances**
- Click **Create notebook instance**:
  - **Name:** `nyc311-modeling`
  - **Instance type:** `ml.t3.medium`
  - **IAM Role:** LabRole
- Click **Create** and wait for status to be "InService" (~5 min)
- Click **Open JupyterLab** when ready

### Step 2: Confirm JupyterLab interface

You should now see:
- **Left sidebar:** File browser, Git, Terminal, etc.
- **Main area:** Editor, notebooks, terminal
- **Top bar:** Menu, create new items, settings

- - -

<a id="github"></a>

## 2. GitHub repo clone

### Clone your repo into JupyterLab

1. **Left sidebar** → Click the **Git icon** (it looks like a branch)

2. Select **Clone a Repository**

3. Paste your GitHub repo URL:
   ```
   https://github.com/YOURUSERNAME/aws-nyc311-yourMSUNetID.git
   # Or whatever your repo URL is
   ```

4. Press **Enter** and authenticate if prompted
   - You'll likely need to set up a GitHub Personal Access Token:
      - Go to GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)
      - Click **Generate new token (classic)** → Select scope: `repo` → Generate token and copy it
      - Paste the token into JupyterLab when prompted for credentials

5. The repo files should now appear in the file browser

### Verify the clone

- Expand the repo folder in the file browser
- You should see: `sql/`, `notes/`, `notebooks/`, `README.md`, etc.

- - -

<a id="test"></a>

## 3. Environment test

### Create a test notebook

1. In the file browser, navigate to the `notebooks/` folder in your cloned repo
1. Then, "**File**" → "**New**" → "**Notebook**"
2. Select **conda_python3** kernel

3. Rename to `data_load_verify.ipynb`

### Run these cells (in your AWS SageMaker notebook) to verify S3 access:

In [ ]:
# Test 1: Import key libraries
import pandas as pd
import boto3
import numpy as np

print("Libraries imported successfully")

In [ ]:
# Test 2: Connect to S3
s3 = boto3.client('s3')
response = s3.list_buckets()
bucket_count = len(response['Buckets'])

print(f"Connected to S3")
print(f"Found {bucket_count} S3 buckets")

# Print your buckets (to identify your modeling data bucket)
for bucket in response['Buckets']:
    print(f"  - {bucket['Name']}")

**Checkpoint:** Did you see your S3 buckets listed? If yes, you're good to move on!

If you got an error, notify the instructor and/or check in with a classmate.

- - - 

<a id="athena"></a>

## 4. Athena data export

### Step 1: Create an Athena table with aggregates

Recall your Day 25 Athena queries. Now you'll create a **"modeling table"** that aggregates data to support a stakeholder question.

We looked at this example together during class, but **you'll review some different options paired with stakeholder questions below and choose one for your modeling efforts in the next class period**.

**Example from class:** Predict high-volume agencies by borough

```sql
-- In Athena query editor:
CREATE TABLE nyc311_db.modeling_data AS
SELECT
  agency,
  borough,
  COUNT(*) AS n_complaints,
  AVG(date_diff('day',
        date_parse(created_date, '%Y-%m-%d %H:%i:%s'),
        CASE WHEN closed_date <> '' THEN date_parse(closed_date, '%Y-%m-%d %H:%i:%s')
             ELSE current_timestamp END)) AS avg_days_to_close,
  NTILE(4) OVER (ORDER BY COUNT(*) DESC) AS volume_quartile
FROM nyc311_db.complaints
GROUP BY agency, borough;
```

This creates a table with features and a target variable for modeling:
- `agency`, `borough`: Features
- `n_complaints`, `avg_days_to_close`: Aggregated features
- `volume_quartile`: Target (1=high volume, 4=low volume)

#### Picking a stakeholder question and modeling table

Now, review the following three stakeholder questions and potential modeling tables. Choose one that interests you for your modeling efforts in the next class period. You can modify the provided queries as needed to fit your question and modeling approach.

When reviewing the stakeholder questions and associated queries, think about:
- What would the **target variable** be? (i.e., what would you be trying to predict)?
- What **features** should be included in the model?
- What **aggregations** or **transformations** are needed to create the features and target variable?
- What exactly is the query doing to create the modeling table? **Review the SQL syntax and logic** to ensure you understand how the table is being created!
- What sort of **modeling approach** would be appropriate for the question and data you've created? (e.g., classification, regression, clustering)

After you choose a question and modeling table, **run the query to create the table in Athena.**

<div class="alert alert-attention">

**Stakeholder question 1:** *"The mayor's office wants to know whether certain types of 311 complaints are likely to be resolved within 3 days — can we build a model to flag fast vs. slow resolutions at intake?"*

The following query produces a row-level dataset with engineered time/category features and a binary target.

```sql
CREATE TABLE nyc311_db.resolution_speed_modeling AS
SELECT
    agency,
    borough,
    problem,
    incident_zip,
    -- Time features from created_date
    day_of_week(date_parse(created_date, '%Y-%m-%d %H:%i:%s'))        AS day_of_week,
    hour(date_parse(created_date, '%Y-%m-%d %H:%i:%s'))                AS hour_of_day,
    -- Broad problem category (reduces high cardinality)
    CASE
        WHEN problem IN ('HEAT/HOT WATER','PLUMBING','WATER LEAK','PAINT/PLASTER',
                         'DOOR/WINDOW','ELECTRIC','GENERAL','UNSANITARY CONDITION') THEN 'housing'
        WHEN problem IN ('Noise - Residential','Noise - Street/Sidewalk',
                         'Noise - Commercial','Noise')                              THEN 'noise'
        WHEN problem IN ('Illegal Parking','Blocked Driveway','Traffic Signal Condition',
                         'Street Condition','Abandoned Vehicle')                    THEN 'traffic'
        WHEN problem IN ('Snow or Ice','Dirty Condition','Water System')             THEN 'sanitation'
        ELSE 'other'
    END AS problem_category,
    -- Binary target: resolved within 3 days?
    CASE
        WHEN closed_date <> ''
         AND date_diff('day',
                date_parse(created_date, '%Y-%m-%d %H:%i:%s'),
                date_parse(closed_date,  '%Y-%m-%d %H:%i:%s')) <= 3
        THEN 1
        ELSE 0
    END AS resolved_quickly
FROM nyc311_db.complaints
WHERE closed_date <> ''
  AND borough IN ('BROOKLYN','QUEENS','BRONX','MANHATTAN','STATEN ISLAND');
```

</div>

<div class="alert alert-attention">

**Stakeholder question 2:** *"Agency directors want a tool that estimates expected resolution time at the moment a complaint is filed, so they can set realistic expectations with residents. What factors drive that time?"*

```sql
CREATE TABLE nyc311_db.resolution_time_modeling AS
SELECT
    agency,
    borough,
    problem,
    incident_zip,
    day_of_week(date_parse(created_date, '%Y-%m-%d %H:%i:%s'))  AS day_of_week,
    hour(date_parse(created_date, '%Y-%m-%d %H:%i:%s'))          AS hour_of_day,
    -- Count of same complaint type filed on same day (proxy for surge volume)
    COUNT(*) OVER (
        PARTITION BY agency, problem,
                     DATE(date_parse(created_date, '%Y-%m-%d %H:%i:%s'))
    ) AS same_day_complaint_volume,
    -- Continuous target: days to close
    date_diff('day',
        date_parse(created_date, '%Y-%m-%d %H:%i:%s'),
        date_parse(closed_date,  '%Y-%m-%d %H:%i:%s')) AS days_to_close
FROM nyc311_db.complaints
WHERE closed_date <> ''
  AND borough IN ('BROOKLYN','QUEENS','BRONX','MANHATTAN','STATEN ISLAND')
  AND date_diff('day',
        date_parse(created_date, '%Y-%m-%d %H:%i:%s'),
        date_parse(closed_date,  '%Y-%m-%d %H:%i:%s')) BETWEEN 0 AND 365;
```

</div>

<div class="alert alert-attention">

**Stakeholder question 3:** *"311 operators want to route incoming calls to the right agency faster. If a complaint comes in at 2am from a residential zip in the Bronx, what type of problem is it most likely to be?"*

```sql
CREATE TABLE nyc311_db.complaint_type_modeling AS
SELECT *
FROM (
    SELECT
        borough,
        incident_zip,
        agency,
        day_of_week(date_parse(created_date, '%Y-%m-%d %H:%i:%s'))  AS day_of_week,
        hour(date_parse(created_date, '%Y-%m-%d %H:%i:%s'))          AS hour_of_day,
        -- Multi-class target (4 meaningful classes)
        CASE
            WHEN problem IN ('HEAT/HOT WATER','PLUMBING','WATER LEAK','PAINT/PLASTER',
                             'DOOR/WINDOW','ELECTRIC','GENERAL','UNSANITARY CONDITION') THEN 'housing'
            WHEN problem IN ('Noise - Residential','Noise - Street/Sidewalk',
                             'Noise - Commercial','Noise')                              THEN 'noise'
            WHEN problem IN ('Illegal Parking','Blocked Driveway','Traffic Signal Condition',
                             'Street Condition','Abandoned Vehicle')                    THEN 'traffic'
            WHEN problem IN ('Snow or Ice','Dirty Condition','Water System')             THEN 'sanitation'
            ELSE NULL
        END AS complaint_category
    FROM nyc311_db.complaints
    WHERE borough IN ('BROOKLYN','QUEENS','BRONX','MANHATTAN','STATEN ISLAND')
)
WHERE complaint_category IS NOT NULL;
```

</div>


### Step 2: Export to S3

Once you've created your modeling table in Athena, you'll export the results to S3 as a CSV file that you can load into SageMaker for modeling.

In Athena:

1. Run your modeling table query (if you haven't already)
2. Using a query (e.g, `SELECT * FROM nyc311_db.modeling_data`), confirm that the results look correct and review the number of rows returned (the number next to "Results")
3. Click **Download results CSV** → Save as `modeling_data.csv` (or something more descriptive)
    - Make sure that you get the full table results in the CSV -- watch out for a `LIMIT` call that might truncate the results.
4. Navigate to S3 → Your bucket → Create or navigate to `modeling/` folder
5. Upload the new CSV file there

**Result:** you should have a new file in your bucket that lives at: `s3://YOUR-BUCKET/modeling/modeling_data.csv`

### Step 3: Save the Athena query

In your local repo, create `sql/athena_to_modeling.sql` and paste your table-generating query along with comments about the features and target variable, i.e.:

```sql
-- Athena query for model data generation (motivated by stakeholder question)
-- Features: (list the features you created in the query here)
-- Target: (list the target variable you created in the query here)
CREATE TABLE ...
```

- - -

<a id="load"></a>

## 5. Load and verify

### Load data in `notebooks/data_load_verify.ipynb`

Now we load the exported CSV into a SageMaker notebook. You should be able to use the notebook you created in Step 3 of the environment test, `data_load_verify.ipynb`.

Once you have the notebook open, use the code in the cells below to load the CSV from S3 and verify that it looks correct.

### Cell 1: Import and load data

For reference, the code below was adapted from this page: https://saturncloud.io/blog/loading-s3-data-into-your-aws-sagemaker-notebook-a-comprehensive-guide/ 

In [ ]:
import pandas as pd
import boto3
from io import StringIO

s3 = boto3.client('s3')
bucket_name = 'your-bucket-name' # replace with your bucket name (make sure you have the account regional suffix)
file_name = 'your-file-name.csv' # make sure to include the path if it's in a folder, e.g. 'modeling/modeling_data.csv'

obj = s3.get_object(Bucket=bucket_name, Key=file_name)
data = obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(data))

### Cell 2: Inspect basic properties

In [ ]:
# First few rows
print("First few rows:")
print(df.head())

# Data types and missing values
print("\n" + "="*50)
print("Data types and missing values:")
print(df.info())

### Cell 3: Summary statistics

In [ ]:
# Summary statistics
print("Summary statistics:")
print(df.describe())

### Cell 4: Target balance and distribution

Since you'll be using the data for modeling, it's crucial to understand the distribution of your target variable. Use appropriate pandas functions to check the balance of classes (for classification) or the distribution (for regression). This will inform your modeling approach and any necessary preprocessing steps.

**Write some code in your notebook to check the balance or distribution of your target variable.**

You can see what this might look like for the example query's target variable, `volume_quartile`, in the code cell below. Make sure you're checking the balance/distribution of the target variable for the stakeholder question and modeling table you chose to work with!

In [ ]:
# Balance check based on the example query, the target variable is `volume_quartile` -- make sure you're checking the right target variable for your modeling table!

# Check target balance
print("Target distribution (volume_quartile):")
print(df['volume_quartile'].value_counts().sort_index())

# As percentage
print("\nTarget distribution (%)")
print((df['volume_quartile'].value_counts() / len(df) * 100).sort_index().round(2))

### Validation checklist

After running these cells, verify:

- Data shape is reasonable (rows = aggregates, columns = features + target)
- No unexpected missing values (unless intended)
- Data types are correct (numeric, string, etc.)
- Target distribution is balanced enough for modeling
- Summary statistics (min, max, mean) make sense

**Questions to ask:**
- Are there outliers that need handling?
- Is the target realistic?
- Do feature values align with your expectations?

- - - 

<a id="plan"></a>

## 6. Modeling plan

### Create `notes/modeling_plan.md`

In your repo (you can do this within JupyterLab in SageMaker, or locally, if you wish), create a file, `modeling_plan.md`, in your `notes` directory to outline your modeling plan using the following template (fill in your details):

```markdown
# NYC 311 Modeling Plan

**Date created:** [today's date]

## Business question
Predict [YOUR QUESTION, e.g., high-volume complaint agencies by borough]

## Data source
- **S3 path:** [YOUR S3 PATH]
- **Records:** [number from df.shape[0]]
- **Athena query:** sql/athena_to_modeling.sql

## Features (update/expand based on your query)
- agency (string)
- borough (string)
- n_complaints (numeric, count of complaints)
- avg_days_to_close (numeric, average resolution time)
- ... (other features)

## Target
- **Name:** [YOUR TARGET VARIABLE, e.g., volume_quartile]
- **Type:** [MODEL TYPE, e.g., Classification (1=high volume, 2-4=lower volume)]
- **Balance/Distribution:** [paste results from your target variable distribution check]

## Modeling approach (update based on your question and data)
- **Baseline:** Logistic regression (interpretable, fast to train)
- **Metrics:** Accuracy, precision, recall
- **Train/test split:** 80/20

## Data quality notes
- [Any missing values, outliers, or issues to watch for]

## Next steps (What you'll work on in the next class period; update/modify based on your plan)
- Train/test split
- Fit baseline logistic regression
- Evaluate and interpret results
```

- - -

<a id="git"></a>

## 7. Git commit and push

### Prepare your repo

Verify you have:
- `notebooks/data_load_verify.ipynb` (with output)
- `sql/athena_to_modeling.sql` (Athena query)
- `notes/modeling_plan.md` (documented plan)

### Commit and push

In the SageMaker terminal (or your local terminal, if you added/edited files locally), run the following commands to commit and push your changes to GitHub:

```bash
cd your-repo
git add notebooks/ sql/ notes/modeling_plan.md
git commit -m "SageMaker setup + data prep + modeling plan"
git push
```

### Verify on GitHub

- Go to GitHub.com → Your repo
- Confirm new/updated files are visible
- Confirm commit message appears in history

- - -

### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page. Go to the In-class assignments section, find the appropriate submission link, and upload it there.

See you in class!

---
© Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.